In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Load the prepared data
df_model = pd.read_csv('../data/processed/data_for_modeling.csv')

# 2. Separate features (X) and target (y)
X = df_model.drop(columns=['churn'])
y = df_model['churn']

# 3. Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train a fast LightGBM model
print("Training LightGBM model...")
model = lgb.LGBMClassifier(
    n_estimators=300, 
    learning_rate=0.05, 
    random_state=42, 
    n_jobs=-1
)
model.fit(X_train, y_train)

# 5. Evaluate the model
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"\n--- ROC-AUC Score: {auc_score:.4f} ---\n")

# 6. Extract and plot Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance.head(20), palette='viridis')
plt.title('Top 20 Most Important Features')
plt.show()

# 7. Print the top 10 features textually
print("Top 10 Most Important Features:")
print(importance.head(10).to_string(index=False))

##  Pilot evaluation (Experiments 1, 2, 3)
Here, we test different models and compare sets of characteristics using a scientific approach.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

# 1. Load Data (Feature Engineering)
df_model = pd.read_csv('../data/processed/data_for_modeling.csv')

# 2. Drop ID (Data Leakage)
if 'Customer_ID' in df_model.columns:
    df_model = df_model.drop(columns=['Customer_ID'])

# 3. Separate X and y
X = df_model.drop(columns=['churn'])
y = df_model['churn']

# 4. Consistent Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Data Ready! Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# ==========================================
# Experiment 3: Model Comparison
# ==========================================
print("--- Experiment 3: Model Comparison ---")

# Scaling for Logistic Regression 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, random_state=42, n_jobs=-1, eval_metric='auc')
}

results = {}
trained_models = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, preds)
    results[name] = auc
    trained_models[name] = model
    print(f"📈 {name} ROC-AUC: {auc:.4f}")

In [ ]:
# ==========================================
# Experiment 2: Feature Set Comparison
# ==========================================
print("\n--- Experiment 2: Feature Selection Comparison ---")

lgb_model = trained_models["LightGBM"]

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# تجربة Top 50, Top 30, Top 15
top_50_features = importance_df.head(50)['Feature'].tolist()
top_30_features = importance_df.head(30)['Feature'].tolist()
top_15_features = importance_df.head(15)['Feature'].tolist()

for top_n, features in [(50, top_50_features), (30, top_30_features), (15, top_15_features)]:
    lgb_sub = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    lgb_sub.fit(X_train[features], y_train)
    preds_sub = lgb_sub.predict_proba(X_test[features])[:, 1]
    
    auc_sub = roc_auc_score(y_test, preds_sub)
    print(f"🌲 LightGBM with Top {top_n} features ROC-AUC: {auc_sub:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc


fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'LightGBM Model (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Falsely predicting churn)', fontsize=12)
plt.ylabel('True Positive Rate (Correctly catching churn)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.show()